# 실험 비교

| Model                     | 1500              | 3000              | 5000              | 10000             |
| ------------------------- | ----------------- | ----------------- | ----------------- | ----------------- |
| **Dense (Deep Learning)** | 0.793 (0.786)     | 0.805 (0.799)     | 0.806 (0.799)     | 0.807 (0.799)     |
| **Naive Bayes**           | 0.691 (0.643)     | 0.687 (0.627)     | 0.673 (0.601)     | 0.657 (0.576)     |
| **Logistic Regression**   | 0.789 (0.762)     | 0.791 (0.766)     | 0.790 (0.765)     | 0.788 (0.761)     |
| **Random Forest**         | 0.776 (0.756)     | 0.770 (0.749)     | 0.769 (0.747)     | 0.754 (0.729)     |
| **Linear SVM**            | **0.829 (0.823)** | **0.829 (0.823)** | **0.829 (0.823)** | **0.830 (0.824)** |
| **Decision Tree**         | 0.684 (0.680)     | 0.706 (0.703)     | 0.699 (0.697)     | 0.697 (0.692)     |


# 회고

제일좋은 모델은 Linear SVM 이었다
- Accuracy = 약 0.83
- F1-Score = 약 0.823

---

실험 결과, Linear SVM이 모든 vocab size에서 가장 높은 성능을 보였으며, 정확도는 약 0.83 이었다.이는 TF-IDF 기반의 고차원, Sparse 텍스트 데이터에서 선형 모델이 다른 모델보단 효과적으로 작동하기 때문으로 생각된다.

딥러닝 모델은 정확도가 약 0.8 수준을 기록하였다. 특히, vocab size가 3000~ 이상부터는 성능 향상이 거의 나타나지 않음을 확인했다.

성능을 비교해보면서, 가장 흥미로웠던 부분은 Naive Bayes 모델은 vocab size가 커질수록 오히려 성능이 점차 감소한다는 부분입니다. 이유를 찾아보니, 단어 수가 증가할수록, 희소성이 증가하기 때문에 naive bayes 모델 가정이 약해져서 임을 새롭게 알게 되었다.

---

- 딥러닝 모델을 사용할 때, Dense, dropout 등 되게 간단하게 모델을 설계하여 실험을 진행했는데, 좀 더 고차원 spasre 텍스트 데이터 특성을 반영할 수 있도록 모델을 다시 체계적으로 설계하여 성능을 비교해보고 싶다.
- 그리고 기존 ML 모델들도 세부적인 하이퍼파라미터 튜닝을 진행하지 않았기 때문에, 더 성능을 높일 수 있는 방법을 모색하여 적용해보고 싶다.
- 속도가 너무 느려서 결국 full vocab size를 진행해보지 못했는데, 이 부분은 나중에 진행해보고 싶다.

In [1]:
import tensorflow
import matplotlib
import seaborn 
import numpy 
import pandas
import sklearn

print(tensorflow.__version__)
print(matplotlib.__version__)
print(seaborn.__version__)
print(numpy.__version__)
print(pandas.__version__)
print(sklearn.__version__)

2.6.0
3.4.3
0.11.2
1.21.4
1.3.3
1.0


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import reuters
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

In [5]:
# 1. Reuters 데이터를 텍스트로 복원하는 함수 (CountVectorizer 사용을 위해)
def decode_reuters(x_data):
    word_index = reuters.get_word_index()
    index_to_word = {i+3: word for word, i in word_index.items()}
    for i, word in enumerate(('<pad>', '<sos>', '<unk>')):
        index_to_word[i] = word

    decoded_docs = []
    for sequence in x_data:
        decoded_docs.append(' '.join([index_to_word.get(i, '?') for i in sequence]))
    return decoded_docs

# 2. 실험 설정
vocab_sizes = [1500, 3000, 5000, 10000] # None은 모든 단어 사용
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, solver='liblinear'),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []

# 3. 실험 루프
for size in vocab_sizes:
    print(f"--- Experimenting with Vocab Size: {size if size else 'All'} ---")

    # 데이터 로드
    (x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=size, test_split=0.2)

    # 텍스트 복원
    x_train_text = decode_reuters(x_train)
    x_test_text = decode_reuters(x_test)

    # TF-IDF 벡터화
    dtmvector = CountVectorizer()
    x_train_dtm = dtmvector.fit_transform(x_train_text)
    x_test_dtm = dtmvector.transform(x_test_text)

    tfidf_transformer = TfidfTransformer()
    x_train_tfidf = tfidf_transformer.fit_transform(x_train_dtm)
    x_test_tfidf = tfidf_transformer.transform(x_test_dtm)

    # 모델별 학습 및 평가
    for name, model in models.items():
        model.fit(x_train_tfidf, y_train)
        predicted = model.predict(x_test_tfidf)

        acc = accuracy_score(y_test, predicted)
        f1 = f1_score(y_test, predicted, average='weighted')

        results.append({
            "Vocab Size": size if size else "Full",
            "Model": name,
            "Accuracy": acc,
            "F1 Score": f1
        })

# 4. 결과 정리
df_results = pd.DataFrame(results)
print("\n[실험 결과 요약]")
print(df_results)

--- Experimenting with Vocab Size: 1500 ---
565248/550378 [==============================] - 0s 0us/step
--- Experimenting with Vocab Size: 3000 ---
--- Experimenting with Vocab Size: 5000 ---
--- Experimenting with Vocab Size: 10000 ---

[실험 결과 요약]
    Vocab Size                Model  Accuracy  F1 Score
0         1500          Naive Bayes  0.691451  0.642744
1         1500  Logistic Regression  0.788513  0.761827
2         1500        Random Forest  0.775601  0.756221
3         1500           Linear SVM  0.829475  0.822643
4         1500        Decision Tree  0.683882  0.679973
5         3000          Naive Bayes  0.687444  0.626617
6         3000  Logistic Regression  0.791184  0.766274
7         3000        Random Forest  0.770258  0.748758
8         3000           Linear SVM  0.829029  0.822879
9         3000        Decision Tree  0.706144  0.703244
10        5000          Naive Bayes  0.673197  0.601250
11        5000  Logistic Regression  0.790294  0.764820
12        5000        

In [6]:
pivot_acc = df_results.pivot(index="Model", columns="Vocab Size", values="Accuracy")
print("\n[Accuracy Pivot Table]")
print(pivot_acc)

pivot_f1 = df_results.pivot(index="Model", columns="Vocab Size", values="F1 Score")
print("\n[F1 Score Pivot Table]")
print(pivot_f1)


[Accuracy Pivot Table]
Vocab Size              1500      3000      5000      10000
Model                                                      
Decision Tree        0.683882  0.706144  0.699466  0.696794
Linear SVM           0.829475  0.829029  0.829029  0.829920
Logistic Regression  0.788513  0.791184  0.790294  0.787622
Naive Bayes          0.691451  0.687444  0.673197  0.656723
Random Forest        0.775601  0.770258  0.768923  0.754230

[F1 Score Pivot Table]
Vocab Size              1500      3000      5000      10000
Model                                                      
Decision Tree        0.679973  0.703244  0.696589  0.691753
Linear SVM           0.822643  0.822879  0.822996  0.823688
Logistic Regression  0.761827  0.766274  0.764820  0.761266
Naive Bayes          0.642744  0.626617  0.601250  0.576447
Random Forest        0.756221  0.748758  0.746734  0.729455


# 딥러닝 모델

In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout

In [10]:
# 1. Reuters 데이터를 텍스트로 복원하는 함수
def decode_reuters(x_data):
    word_index = reuters.get_word_index()
    index_to_word = {i+3: word for word, i in word_index.items()}
    for i, word in enumerate(('<pad>', '<sos>', '<unk>')):
        index_to_word[i] = word

    decoded_docs = []
    for sequence in x_data:
        decoded_docs.append(' '.join([index_to_word.get(i, '?') for i in sequence]))
    return decoded_docs

# 2. 실험할 vocab size
vocab_sizes = [1500, 3000, 5000, 10000]

# 3. 결과 저장용
dl_results = []

# 4. 실험 루프
for size in vocab_sizes:
    print(f"\n--- Deep Learning Experiment with Vocab Size: {size if size else 'Full'} ---")

    # 데이터 로드
    (x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=size, test_split=0.2)

    # 텍스트 복원
    x_train_text = decode_reuters(x_train)
    x_test_text = decode_reuters(x_test)

    # CountVectorizer + TF-IDF
    dtmvector = CountVectorizer()
    x_train_dtm = dtmvector.fit_transform(x_train_text)
    x_test_dtm = dtmvector.transform(x_test_text)

    tfidf_transformer = TfidfTransformer()
    x_train_tfidf = tfidf_transformer.fit_transform(x_train_dtm)
    x_test_tfidf = tfidf_transformer.transform(x_test_dtm)

    # keras Dense 입력을 위해 dense array로 변환
    x_train_dense = x_train_tfidf.toarray()
    x_test_dense = x_test_tfidf.toarray()

    # 입력 차원 자동 설정
    input_dim = x_train_dense.shape[1]

    # Dense 모델 생성
    inputs = Input(shape=(input_dim,))
    x = Dense(512, activation='relu')(inputs)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(46, activation='softmax')(x)

    dense_model = Model(inputs=inputs, outputs=outputs)

    dense_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    dense_model.summary()

    # 학습
    dense_model.fit(
        x_train_dense, y_train,
        epochs=10,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    # 예측
    y_pred_prob = dense_model.predict(x_test_dense)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # 평가
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    dl_results.append({
        "Vocab Size": size if size else "Full",
        "Model": "Dense",
        "Accuracy": acc,
        "F1 Score": f1
    })

# 5. 결과 출력
df_dl_results = pd.DataFrame(dl_results)
print("\n[딥러닝 실험 결과 요약]")
print(df_dl_results)


--- Deep Learning Experiment with Vocab Size: 1500 ---
Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 1455)]            0         
_________________________________________________________________
dense_3 (Dense)              (None, 512)               745472    
_________________________________________________________________
dropout_2 (Dropout)          (None, 512)               0         
_________________________________________________________________
dense_4 (Dense)              (None, 128)               65664     
_________________________________________________________________
dropout_3 (Dropout)          (None, 128)               0         
_________________________________________________________________
dense_5 (Dense)              (None, 46)                5934      
Total params: 817,070
Trainable params: 817,070
Non-trainable params:

Epoch 1/10
225/225 [==============================] - 1s 5ms/step - loss: 1.6884 - accuracy: 0.6168 - val_loss: 1.0986 - val_accuracy: 0.7457
Epoch 2/10
225/225 [==============================] - 1s 3ms/step - loss: 0.8120 - accuracy: 0.8103 - val_loss: 0.8665 - val_accuracy: 0.7986
Epoch 3/10
225/225 [==============================] - 1s 3ms/step - loss: 0.4668 - accuracy: 0.8906 - val_loss: 0.8543 - val_accuracy: 0.8114
Epoch 4/10
225/225 [==============================] - 1s 3ms/step - loss: 0.2887 - accuracy: 0.9321 - val_loss: 0.8620 - val_accuracy: 0.8197
Epoch 5/10
225/225 [==============================] - 1s 3ms/step - loss: 0.2114 - accuracy: 0.9479 - val_loss: 0.8654 - val_accuracy: 0.8125
Epoch 6/10
225/225 [==============================] - 1s 4ms/step - loss: 0.1753 - accuracy: 0.9525 - val_loss: 0.8739 - val_accuracy: 0.8164
Epoch 7/10
225/225 [==============================] - 1s 3ms/step - loss: 0.1438 - accuracy: 0.9580 - val_loss: 0.9351 - val_accuracy: 0.8086
Epoch 